# ProFET-style biophysical features (Biopython) — batch, per gene family

Fixes vs. the original script:
- **Adds the 4 missing features**: `net_charge_pH_5`, `net_charge_pH_7`, `net_charge_pH_9` (Biopython's `charge_at_pH()`, already built in, just wasn't called), `aliphatic_index` (not a Biopython method — computed manually from the Ikai 1980 formula using `amino_acids_percent`).
- **No more whole-run crash on one bad sequence.** The original died on the first `X`/`B`/`Z`/`U`/`O`/`J` residue it hit, with zero output saved. Each feature is now computed independently — verified directly that `isoelectric_point`, `aromaticity`, `charge_at_pH`, and `amino_acids_percent` (needed for aliphatic index) all survive an `X` in the sequence; only `molecular_weight`, `instability_index`, and `gravy` actually fail on it. So a sequence with one bad residue now loses 3 columns, not the entire batch.
- **Renamed `length` → `sequence_length`** to match what you actually asked for, and dropped the unused `IsoelectricPoint` import.
- **Restructured to match `arg_pfeature_batch_extraction.ipynb`**: batches over per-family FASTA files in a folder (not one pooled `all_card_proteins.fasta`), keyed the same way — `sequence_id` from the FASTA header, `family` from the filename — so this output is directly joinable with the Pfeature feature tables on those two columns.

**Before running:** set `INPUT_DIR` below to the same per-family FASTA directory used for the Pfeature notebook.

In [3]:
import shutil
import numpy as np
import pandas as pd
from pathlib import Path
from Bio import SeqIO
from Bio.SeqUtils import molecular_weight, ProtParam
from tqdm.auto import tqdm

import Bio
print(f"biopython {Bio.__version__}")
print(f"pandas {pd.__version__}")
assert hasattr(ProtParam.ProteinAnalysis, "charge_at_pH"), "This Biopython version lacks charge_at_pH() -- upgrade biopython"


biopython 1.86
pandas 2.3.3


In [4]:
# --- point this at the SAME per-family protein FASTA directory as the Pfeature notebook ---
INPUT_DIR = Path("/home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/CARD_DATA/card-data/refined_details/Final_Results/Drug_Class_Fastas/protein")

OUTPUT_DIR = Path("//home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/CARD_DATA/card-data/refined_details/Final_Results/Drug_Class_Fastas_profet")   # sits next to results/pfeature_features
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FASTA_GLOBS = ["*.fasta", "*.fa", "*.faa"]
PH_VALUES = [5.0, 7.0, 9.0]     # matches net_charge_pH_5 / _7 / _9
MIN_SEGMENT_LEN = 5             # same threshold as the original script
STANDARD_AA = set("ACDEFGHIKLMNPQRSTVWY")


### Notes

- **Error handling is per-feature, not per-sequence or per-file.** A non-standard residue kills `molecular_weight_da`, `instability_index`, and `gravy` for that specific sequence (those NaN out) but leaves `isoelectric_point`, `aromaticity`, the three `net_charge_pH_*` columns, and `aliphatic_index` intact — confirmed directly which calls survive an `X` before writing this. Nothing here decides *for* you whether to exclude flagged sequences afterward; the pre-flight scan just tells you where they are, same as the Pfeature notebook's `scan_nonstandard_residues`.
- **Aliphatic index isn't a Biopython method.** Computed here as `A% + 2.9*V% + 3.9*(I%+L%)` (Ikai, 1980) from `analysis.amino_acids_percent`, which is a plain dict attribute already on a 0–100 scale — not a method call, and no rescaling needed.
- Segment logic (N/M/C thirds, `>=5` residue minimum) is unchanged from your original — that part had no bugs.

In [5]:
def discover_family_fastas(input_dir: Path):
    files = set()
    for pattern in FASTA_GLOBS:
        files.update(input_dir.glob(pattern))
    return sorted(files)

def scan_nonstandard_residues(fasta_path: Path):
    flagged = {}
    for record in SeqIO.parse(str(fasta_path), "fasta"):
        extra = set(str(record.seq).upper()) - STANDARD_AA
        if extra:
            flagged[record.id] = sorted(extra)
    return flagged

def aliphatic_index(percent_dict: dict) -> float:
    # Ikai (1980). percent_dict values are already 0-100 (Biopython convention).
    return (percent_dict.get("A", 0)
            + 2.9 * percent_dict.get("V", 0)
            + 3.9 * (percent_dict.get("I", 0) + percent_dict.get("L", 0)))

def safe_call(fn, *args):
    """Run one feature computation; NaN out on failure instead of killing the batch."""
    try:
        return fn(*args)
    except Exception:
        return np.nan

def protein_features(sequence: str, prefix: str = "") -> dict:
    sequence = str(sequence).upper()
    if not sequence:
        return {}
    analysis = ProtParam.ProteinAnalysis(sequence)
    feats = {
        f"{prefix}sequence_length": len(sequence),
        f"{prefix}molecular_weight_da": safe_call(molecular_weight, sequence, "protein"),
        f"{prefix}isoelectric_point": safe_call(analysis.isoelectric_point),
        f"{prefix}aromaticity": safe_call(analysis.aromaticity),
        f"{prefix}instability_index": safe_call(analysis.instability_index),
        f"{prefix}gravy": safe_call(analysis.gravy),
    }
    for ph in PH_VALUES:
        feats[f"{prefix}net_charge_pH_{int(ph)}"] = safe_call(analysis.charge_at_pH, ph)
    try:
        feats[f"{prefix}aliphatic_index"] = aliphatic_index(analysis.amino_acids_percent)
    except Exception:
        feats[f"{prefix}aliphatic_index"] = np.nan
    return feats

def segment_features(sequence: str) -> dict:
    sequence = str(sequence).upper()
    length = len(sequence)
    cut_1 = length // 3
    cut_2 = 2 * length // 3
    segments = {"N_": sequence[:cut_1], "M_": sequence[cut_1:cut_2], "C_": sequence[cut_2:]}
    result = {}
    for prefix, segment in segments.items():
        if len(segment) >= MIN_SEGMENT_LEN:
            result.update(protein_features(segment, prefix=prefix))
    return result


In [6]:
family_fastas = discover_family_fastas(INPUT_DIR)
print(f"Found {len(family_fastas)} family FASTA file(s) in {INPUT_DIR}")

nonstandard_report = {}
for fp in family_fastas:
    flagged = scan_nonstandard_residues(fp)
    if flagged:
        nonstandard_report[fp.name] = flagged

if nonstandard_report:
    print(f"\n{len(nonstandard_report)} file(s) contain non-standard residues -- not blocking, "
          f"molecular_weight_da/instability_index/gravy will be NaN for those specific sequences:")
    for fname, seqs in nonstandard_report.items():
        print(f"  {fname}: {len(seqs)} sequence(s), e.g. {list(seqs.items())[:2]}")
else:
    print("No non-standard residues detected.")


Found 39 family FASTA file(s) in /home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/CARD_DATA/card-data/refined_details/Final_Results/Drug_Class_Fastas/protein

3 file(s) contain non-standard residues -- not blocking, molecular_weight_da/instability_index/gravy will be NaN for those specific sequences:
  cephalosporin.fasta: 2 sequence(s), e.g. [('ARO:3004647|gb|AHF82023.1|ARO:3004647|AQU-2', ['X']), ('ARO:3004648|gb|AHF82024.1|ARO:3004648|AQU-3', ['X'])]
  rifamycin_antibiotic.fasta: 1 sequence(s), e.g. [('ARO:3000444|gb|AIA08936.1|ARO:3000444|rphA', ['X'])]
  tetracycline_antibiotic.fasta: 2 sequence(s), e.g. [('ARO:3000573|gb|ACS83748.1|ARO:3000573|tet(43)', ['X']), ('ARO:3002894|gb|AAR96051.1|ARO:3002894|otr(C)', ['X'])]


In [7]:
for fp in tqdm(family_fastas, desc="families"):
    records = []
    for record in SeqIO.parse(str(fp), "fasta"):
        row = {"sequence_id": record.id}
        seq = str(record.seq)
        row.update(protein_features(seq))
        row.update(segment_features(seq))
        records.append(row)
    df = pd.DataFrame(records)
    df.to_csv(OUTPUT_DIR / f"{fp.stem}_profet_features.csv", index=False)

print(f"Per-family feature tables written to {OUTPUT_DIR}/<family>_profet_features.csv")


families: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 39/39 [00:05<00:00,  7.07it/s]

Per-family feature tables written to //home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/CARD_DATA/card-data/refined_details/Final_Results/Drug_Class_Fastas_profet/<family>_profet_features.csv


In [8]:
# NaN summary -- which columns actually got hit, and how often, across every family just processed
family_tables = sorted(OUTPUT_DIR.glob("*_profet_features.csv"))
total_rows = 0
nan_counts = pd.Series(dtype=int)

for ft in family_tables:
    df = pd.read_csv(ft)
    total_rows += len(df)
    nan_counts = nan_counts.add(df.isna().sum(), fill_value=0)

affected = nan_counts[nan_counts > 0].sort_values(ascending=False)
print(f"{total_rows} sequences processed across {len(family_tables)} families.")
if len(affected):
    print(f"\nColumns with at least one NaN (feature-level failures, not dropped rows):")
    for col, cnt in affected.items():
        print(f"  {col}: {int(cnt)} / {total_rows}")
else:
    print("No NaNs -- every feature computed cleanly for every sequence.")


12913 sequences processed across 39 families.

Columns with at least one NaN (feature-level failures, not dropped rows):
  C_gravy: 5 / 12913
  C_instability_index: 5 / 12913
  C_molecular_weight_da: 5 / 12913
  instability_index: 5 / 12913
  gravy: 5 / 12913
  molecular_weight_da: 5 / 12913
  N_gravy: 1 / 12913
  N_instability_index: 1 / 12913
  N_molecular_weight_da: 1 / 12913


In [9]:
# Combined table across all families -- same pattern as arg_pfeature_batch_extraction.ipynb,
# joinable with the Pfeature master table on (family, sequence_id)
family_tables = sorted(OUTPUT_DIR.glob("*_profet_features.csv"))
if family_tables:
    frames = []
    for ft in family_tables:
        df = pd.read_csv(ft)
        fam_col = pd.Series([ft.stem.replace("_profet_features", "")] * len(df), name="family")
        frames.append(pd.concat([fam_col, df], axis=1))
    master = pd.concat(frames, axis=0, ignore_index=True)
    master.to_csv(OUTPUT_DIR / "all_families_profet_features.csv", index=False)
    print(f"Master table: {master.shape[0]} sequences x {master.shape[1]-2} features across {len(family_tables)} families")
    print(f"-> {OUTPUT_DIR / 'all_families_profet_features.csv'}")
else:
    print("No per-family tables found -- check the cells above.")


Master table: 12913 sequences x 40 features across 39 families
-> //home/user/Pranjal-11-12-2025/ESKAPE_ARG_Family/CARD_DATA/card-data/refined_details/Final_Results/Drug_Class_Fastas_profet/all_families_profet_features.csv
